This project implements a hybrid solution combining **Google Document AI** OCR for high-precision document reading and the **Vertex AI (Gemini 2.0 Flash)** LLM for intelligent analysis and structured data extraction.

# Imports


In [ ]:
import os
import json
import io
import time
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
from PyPDF2 import PdfReader, PdfWriter

# --- CONFIGURATION ---
PROJECT_ID = "your-project-id"
LOCATION = "eu"
PROCESSOR_ID = "your-processor-id"

INPUT_FOLDER = r"C:\Path\To\Your\Input\Folder"
OUTPUT_FILE = r"C:\Path\To\Your\Output\results.json"

print("Configuration loaded.")

# Init

In [ ]:
vertexai.init(
    project=PROJECT_ID,
    location="europe-central2",
    api_endpoint="europe-central2-aiplatform.googleapis.com"
)
model = GenerativeModel("gemini-2.0-flash")
print(f"Model initialised : {model}")

# LLM Prompt

In [ ]:
EXTRACTION_PROMPT = """
You are an expert in logistics. Extract data if and only if it is a valid transport invoice.
Rules:
1. Reject if 'PARKING_COMPANY' is mentioned.
2. Issuer must be: 'FORWARDER_A', 'FORWARDER_B', 'FORWARDER_C', 'FORWARDER_D'.
3. Reject Customs or Food invoices.

Output JSON:
[{"classification": "original_invoice", "invoice_number": "string", "invoice_details": {"freight_forwarder": "string", "date": "DD/MM/YYYY", "destination": "City"}}]
"""
print("Prompt defini.")

# Utility Functions

In [ ]:
def get_docai_text(client, processor_name, pdf_content):
    raw_document = documentai.RawDocument(content=pdf_content, mime_type="application/pdf")
    request = documentai.ProcessRequest(name=processor_name, raw_document=raw_document)
    result = client.process_document(request=request)
    return result.document.text

def analyze_with_llm(ocr_text):
    if not ocr_text or len(ocr_text.strip()) < 10: return []
    full_prompt = f"{EXTRACTION_PROMPT}\n\n--- TEXT ---\n{ocr_text}"
    config = GenerationConfig(temperature=0.1, response_mime_type="application/json")
    try:
        res = model.generate_content(full_prompt, generation_config=config)
        return json.loads(res.text.strip(" `").replace("json", ""))
    except: return []

print("Fonctions chargees.")

# Pipeline

In [ ]:
def process_document_pipeline(file_path):
    opts = ClientOptions(api_endpoint=f"{LOCATION}-documentai.googleapis.com")
    client = documentai.DocumentProcessorServiceClient(client_options=opts)
    name = client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)
    results = []
    with open(file_path, "rb") as f:
        reader = PdfReader(f)
        for i in range(len(reader.pages)):
            writer = PdfWriter()
            writer.add_page(reader.pages[i])
            buf = io.BytesIO()
            writer.write(buf)
            try:
                text = get_docai_text(client, name, buf.getvalue())
                data = analyze_with_llm(text)
                for item in data:
                    if item.get("classification") == "original_invoice":
                        item["source_page"] = i + 1
                        results.append(item)
            except: continue
            time.sleep(0.5)
    return results
print("Pipeline pret.")

# Execution

In [ ]:
if os.path.exists(INPUT_FOLDER):
    summary = []
    files = [f for f in os.listdir(INPUT_FOLDER) if f.lower().endswith('.pdf')]
    for filename in files:
        res = process_document_pipeline(os.path.join(INPUT_FOLDER, filename))
        for entry in res:
            entry["file_name"] = filename
            summary.append(entry)
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=4)
    print("Termine.")

# Conclusion 

Thanks to the synergy between the extraction power of **Document AI** and the analytical intelligence of **Vertex AI (Gemini)**, this pipeline delivers **perfect detection** and flawless classification of the processed documents.